Extract covid-19 data from disease.sh - Open Disease Data API and store the data in a database.

Fetch the data from the API

In [18]:
import requests
from datetime import datetime

def fetch_covid_data(api_url):
    response = requests.get(api_url)
    if response.status_code == 200:
        data = response.json()
        covid_data = []

        # Process the data
        for date, cases in data["cases"].items():
            covid_data.append({
                "date": datetime.strptime(date, "%m/%d/%y").strftime("%Y-%m-%d"),
                "cases": cases,
                "deaths": data["deaths"].get(date, 0),  # Handle missing values
                "recovered": data["recovered"].get(date, 0)  # Handle missing values
            })
        print("Data fetched successfully.")
        return covid_data
    else:
        print("Failed to fetch data:", response.json())
        exit()

API_URL = "https://disease.sh/v3/covid-19/historical/all?lastdays=all"
raw_covid_data = fetch_covid_data(API_URL)


Data fetched successfully.


Clean Data

In [19]:
def clean_covid_data(raw_data):
    cleaned_data = []

    for record in raw_data:
        # Ensure no negative values (if applicable)
        record["cases"] = max(0, record["cases"])
        record["deaths"] = max(0, record["deaths"])
        record["recovered"] = max(0, record["recovered"])

        # Handle inconsistencies (e.g., deaths cannot exceed cases)
        if record["deaths"] > record["cases"]:
            record["deaths"] = record["cases"]

        # Add cleaned record to the list
        cleaned_data.append(record)

    print("Data cleaned successfully.")
    return cleaned_data

covid_data = clean_covid_data(raw_covid_data)


Data cleaned successfully.


Store the Fetch Data in Database

In [20]:
import psycopg2

#connect to database
conn = psycopg2.connect(
    dbname="covid",
    user='postgres',
    password="KARU55bime22",
    host="localhost", 
    port="5432"
)

cursor = conn.cursor()

conn.commit()

# Insert data to the table(already create the table)
insert_query = """
INSERT INTO covid_d (date, cases, deaths, recovered)
VALUES (%s, %s, %s, %s)
ON CONFLICT (date) DO NOTHING;
"""

for record in covid_data:
    cursor.execute(insert_query, (
        record["date"],
        record["cases"],
        record["deaths"],
        record["recovered"]
    ))

conn.commit()

# Query the database to check the contents
select_query = "SELECT * FROM covid_d LIMIT 5;"
cursor.execute(select_query)
result = cursor.fetchall()

# Close the connection
cursor.close()
conn.close()
print("Data successfully inserted into the database.",result)


Data successfully inserted into the database. [(1, datetime.date(2020, 1, 22), 557, 17, 30), (2, datetime.date(2020, 1, 23), 657, 18, 32), (3, datetime.date(2020, 1, 24), 944, 26, 39), (4, datetime.date(2020, 1, 25), 1437, 42, 42), (5, datetime.date(2020, 1, 26), 2120, 56, 56)]


Extract live weather data for a specific country from OpenWeatherMap API and store the data in a database.

Fetch the data from API

In [21]:
import requests
import psycopg2
from datetime import datetime

API_KEY = "d6dbc559887d2c7a79a59caad00d9439"
BASE_URL = "http://api.openweathermap.org/data/2.5/weather"
city_name = "London" # country name

# Fetch weather data from OpenWeatherMap API
params = {
    "q": city_name,
    "appid": API_KEY,
    "units": "metric" 
}
response = requests.get(BASE_URL, params=params)

if response.status_code == 200:
    data = response.json()
    # Extract relevant fields
    weather_data = {
        "city": data["name"],
        "datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "temperature": data["main"]["temp"],
        "humidity": data["main"]["humidity"],
        "wind_speed": data["wind"]["speed"] * 3.6,  # Convert m/s to km/h
        "description": data["weather"][0]["description"]
    }
    print("Fetch data successfully")     
else:
    print("Failed to fetch data:", response.json())
    exit()



Fetch data successfully


Store the fetched data in database

In [22]:
#connect to database
conn = psycopg2.connect(
    dbname="weather",  
    user='postgres', 
    password="KARU55bime22", 
    host="localhost", 
    port="5432"  
)

cursor = conn.cursor()

# Insert weather data into the table
insert_query = """
INSERT INTO api_data (city, datetime, temperature, humidity, wind_speed, description)
VALUES (%s, %s, %s, %s, %s, %s)
"""
cursor.execute(insert_query, (
    weather_data["city"],
    weather_data["datetime"],
    weather_data["temperature"],
    weather_data["humidity"],
    weather_data["wind_speed"],
    weather_data["description"]
))
conn.commit()

# Query the database to check the contents
select_query = "SELECT * FROM api_data LIMIT 5;"
cursor.execute(select_query)
result = cursor.fetchall()

# Close the connection
cursor.close()
conn.close()
print("Data successfully inserted into the database.")


Data successfully inserted into the database.


Extract live weather data for a several countries from OpenWeatherMap API and store the data in a database.

In [23]:
import requests
import psycopg2
from datetime import datetime
import time

# API key and base URL for OpenWeatherMap
API_KEY = "d6dbc559887d2c7a79a59caad00d9439"
BASE_URL = "http://api.openweathermap.org/data/2.5/weather"

# Sample list of cities and their countries
cities = [
    {"name": "New York", "country": "USA"},
    {"name": "London", "country": "UK"},
    {"name": "Tokyo", "country": "Japan"},
    {"name": "Delhi", "country": "India"},
    {"name": "Sydney", "country": "Australia"}
]

# connect to database
conn = psycopg2.connect(
    dbname="weather", 
    user="postgres", 
    password="KARU55bime22",
    host="localhost", 
    port="5432"  
)
cursor = conn.cursor()

# Loop through the city list and fetch weather data
for city in cities:
    params = {
        "q": city["name"],
        "appid": API_KEY,
        "units": "metric"
    }
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        data = response.json()
        # Extract relevant fields
        weather_data = {
            "city": data["name"],
            "country": city["country"],
            "datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "temperature": data["main"]["temp"],
            "humidity": data["main"]["humidity"],
            "wind_speed": data["wind"]["speed"] * 3.6,  # Convert m/s to km/h
            "description": data["weather"][0]["description"]
        }
        
        # Insert weather data into the table
        insert_query = """
        INSERT INTO api_d (city, country, datetime, temperature, humidity, wind_speed, description)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        """
        cursor.execute(insert_query, (
            weather_data["city"],
            weather_data["country"],
            weather_data["datetime"],
            weather_data["temperature"],
            weather_data["humidity"],
            weather_data["wind_speed"],
            weather_data["description"]
        ))
        conn.commit()
        print(f"Inserted data for {city['name']}")
    else:
        print(f"Failed to fetch data for {city['name']}: {response.status_code}")

    # Pause to avoid hitting the rate limit
    time.sleep(1)

# Query the database to check the contents
select_query = "SELECT * FROM api_data LIMIT 10;"
cursor.execute(select_query)
result = cursor.fetchall()

# Close the connection
cursor.close()
conn.close()

# Display the result
print("Inserted Weather Data:", result)


Inserted data for New York
Inserted data for London
Inserted data for Tokyo
Inserted data for Delhi
Inserted data for Sydney
Inserted Weather Data: [('London', datetime.datetime(2024, 11, 28, 16, 12, 46), 3.11, 90, 1.836, 'overcast clouds'), ('London', datetime.datetime(2024, 11, 30, 13, 25, 39), 11.35, 91, 4.824000000000001, 'overcast clouds'), ('London', datetime.datetime(2024, 11, 30, 13, 26, 15), 11.35, 91, 4.824000000000001, 'overcast clouds'), ('London', datetime.datetime(2024, 11, 30, 13, 26, 27), 11.35, 91, 4.824000000000001, 'overcast clouds'), ('London', datetime.datetime(2024, 12, 2, 9, 23, 12), 11.49, 86, 14.832, 'overcast clouds'), ('London', datetime.datetime(2024, 12, 2, 10, 1, 7), 11.46, 86, 16.668, 'broken clouds'), ('London', datetime.datetime(2024, 12, 3, 15, 22, 22), 5.67, 87, 5.5440000000000005, 'few clouds'), ('London', datetime.datetime(2024, 12, 3, 15, 24, 1), 5.67, 87, 5.5440000000000005, 'few clouds'), ('London', datetime.datetime(2024, 12, 3, 15, 24, 21), 5.6

In [33]:
import requests
from datetime import datetime

def fetch_weather_data(city_name, api_key):
    BASE_URL = "http://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city_name,
        "appid": api_key,
        "units": "metric"
    }
    
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        weather_data = {
            "city": data["name"],
            "datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "temperature": data["main"]["temp"],
            "humidity": data["main"]["humidity"],
            "wind_speed": data["wind"]["speed"] * 3.6,  # Convert m/s to km/h
            "description": data["weather"][0]["description"]
        }
        print("Weather data fetched successfully.")
        return weather_data
    else:
        print("Failed to fetch weather data:", response.json())
        exit()

API_KEY = "d6dbc559887d2c7a79a59caad00d9439"
city_name = "London"
weather_data = fetch_weather_data(city_name, API_KEY)


Weather data fetched successfully.


Clean the Data

In [36]:
def clean_weather_data(weather_data):
    # Ensure temperature, humidity, and wind_speed are non-negative
    weather_data["temperature"] = max(weather_data["temperature"], 0)
    weather_data["humidity"] = max(weather_data["humidity"], 0)
    weather_data["wind_speed"] = max(weather_data["wind_speed"], 0)

    # Check if description is missing or empty
    if not weather_data["description"]:
        weather_data["description"] = "No description available"

    # Ensure the datetime is in the right format (it already is, but it's good to check)
    try:
        datetime.strptime(weather_data["datetime"], "%Y-%m-%d %H:%M:%S")
    except ValueError:
        weather_data["datetime"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # Default to current time if invalid
    
    print("Data cleaned successfully.")
    return weather_data


In [39]:
import psycopg2

#connect to database
conn = psycopg2.connect(
    dbname="weather",  
    user='postgres', 
    password="KARU55bime22", 
    host="localhost", 
    port="5432"  
)

cursor = conn.cursor()

# Insert weather data into the table
insert_query = """
INSERT INTO api_data (city, datetime, temperature, humidity, wind_speed, description)
VALUES (%s, %s, %s, %s, %s, %s)
"""
cursor.execute(insert_query, (
    weather_data["city"],
    weather_data["datetime"],
    weather_data["temperature"],
    weather_data["humidity"],
    weather_data["wind_speed"],
    weather_data["description"]
))
conn.commit()

# Query the database to check the contents
select_query = "SELECT * FROM api_data LIMIT 5;"
cursor.execute(select_query)
result = cursor.fetchall()

# Close the connection
cursor.close()
conn.close()
print("Data successfully inserted into the database.")


Data successfully inserted into the database.
